In [0]:
# secrets_path='/Volumes/databricks_learning/custom/secrets'
# bootstrap_servers ='kafka-instance-bipin-data-engineering-project-1.j.aivencloud.com:10670'
# topic='events'
# bronze_checkpoint_path='/Volumes/databricks_learning/custom/checkpoint/bronze_checkpoint'
# bronze_table_name='databricks_learning.custom.events_bronze'

secrets_path=''
bootstrap_servers =''
topic=''
bronze_checkpoint_path=''
bronze_table_name=''

dbutils.widgets.text(name='bootstrap_servers',defaultValue=bootstrap_servers, label='bootstrap_servers')
dbutils.widgets.text(name='topic',defaultValue=topic, label='topic')
dbutils.widgets.text(name='bronze_checkpoint_path',defaultValue=bronze_checkpoint_path, label='bronze_checkpoint_path')
dbutils.widgets.text(name='bronze_table_name',defaultValue=bronze_table_name, label='bronze_table_name')
dbutils.widgets.text(name='secrets_path',defaultValue=secrets_path, label='secrets_path')

bootstrap_servers = dbutils.widgets.get('bootstrap_servers')
topic = dbutils.widgets.get('topic')
bronze_checkpoint_path = dbutils.widgets.get('bronze_checkpoint_path')
bronze_table_name = dbutils.widgets.get('bronze_table_name')
secrets_path = dbutils.widgets.get('secrets_path')

In [0]:
kafka_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .option("kafka.security.protocol", "SSL")
        .option(
            "kafka.ssl.truststore.type",
            "PEM"
        )
        .option(
            "kafka.ssl.keystore.type",
            "PEM"
        )
        .option(
            "kafka.ssl.truststore.certificates",
            open(f"{secrets_path}/ca_certificate.txt").read()
        )
        .option(
            "kafka.ssl.keystore.certificate.chain",
            open(f"{secrets_path}/access_certificate.txt").read()
        )
        .option(
            "kafka.ssl.keystore.key",
            open(f"{secrets_path}/access_key.txt").read()
        )
        .load()
)

In [0]:
avro_schema = """
{
"type": "record",
"name": "PersonEvent",
"fields": [
{
"name": "name",
"type": "string"
}
]
}"""

In [0]:
query = (kafka_df.writeStream\
        .format("delta")\
        .outputMode("append")\
        .option("checkpointLocation", bronze_checkpoint_path)\
        .trigger(availableNow=True)\
        .table(bronze_table_name))

In [0]:
%skip
query.awaitTermination()
print(query.status)
print(query.lastProgress)

In [0]:
%skip
%sql
select * from databricks_learning.custom.events_bronze;